In [ ]:
import os
import math
import glob
import time  # Added to track time
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.spatial import KDTree
print("Testing imports...", flush=True)
import cv2
print("cv2 imported!", flush=True)
import skimage
print("skimage imported!", flush=True)

print("--- DIAGNOSTICS STARTED ---")
start_total = time.time()

images = []
final_list = []

def functionname(img1, img2, distance):
    coords1 = np.argwhere(img1 > 0)
    coords2 = np.argwhere(img2 > 0)
    if len(coords1) == 0 or len(coords2) == 0:
        return []
    tree = KDTree(coords2)
    matches = tree.query_ball_point(coords1, r=distance)
    vectors_list = []
    for idx1, match_indices in enumerate(matches):
        x, y = coords1[idx1]
        for idx2 in match_indices:
            newx, newy = coords2[idx2]
            if newx == x and newy == y:
                continue
            star_distance = math.sqrt((y - newy)**2 + (x - newx)**2)
            star_angle = math.atan2((newy - y), (newx - x))
            vectors_list.append([x, y, newx, newy, star_angle, star_distance])
    return vectors_list

def paint_sky(matched_vectors, dilation):
    image1, image2, image3, image4 = [np.zeros((1024, 1024), dtype=np.uint8) for _ in range(4)]
    coordinates1, coordinates2, coordinates3, coordinates4 = [], [], [], []
    for span in matched_vectors:
        coordinates1.append((int(span[0][0]), int(span[0][1])))
        coordinates2.append((int(span[1][0]), int(span[1][1])))
        coordinates3.append((int(span[2][0]), int(span[2][1])))
        coordinates4.append((int(span[2][2]), int(span[2][3])))
    for coord in coordinates1: image1[coord[1], coord[0]] = 1
    for coord in coordinates2: image2[coord[1], coord[0]] = 1
    for coord in coordinates3: image3[coord[1], coord[0]] = 1
    for coord in coordinates4: image4[coord[1], coord[0]] = 1
    kernel = np.ones((dilation, dilation), np.uint8)
    return (cv2.dilate(image1, kernel, iterations=1),
            cv2.dilate(image2, kernel, iterations=1),
            cv2.dilate(image3, kernel, iterations=1),
            cv2.dilate(image4, kernel, iterations=1))

def matrix_movie(frames, output_file, fps, scalar):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_file, fourcc, fps, (1024, 1024))
    for comet in frames:
        comet_bgr = cv2.cvtColor(comet, cv2.COLOR_GRAY2BGR) * scalar
        out.write(np.clip(comet_bgr, 0, 255).astype(np.uint8))
    out.release()

# 1. Image Loading Check
print("[1/5] Loading images and mask...")
try:
    Mask = Image.open('Images/mask.png')
    os.chdir('Images/images')
    SOHO_images = sorted(glob.glob('*.gif'))
    for file in SOHO_images:
        images.append(file)
    MaskArray = np.array(Mask)
    if len(MaskArray.shape) == 3:
        MaskArray = np.mean(MaskArray, axis=2)
    print(f"      Successfully found {len(images)} images.")
except Exception as e:
    print(f"❌ ERROR in Step 1: {e}")

# 2. Image Processing Loop Check
print("[2/5] Processing image frames (morphology & skeletonize)...")
for i in images[5:10]:
    t0 = time.time()
    img = Image.open(i)
    imgArray = np.array(img)
    binary = imgArray > 160
    
    mask2 = skimage.morphology.remove_small_objects(binary, min_size=20)
    mask2 = ~skimage.morphology.dilation(mask2, skimage.morphology.disk(20))
    hold = binary & mask2
    comets_only = hold * MaskArray * imgArray
    
    # Timing skeletonize specifically because it can hang on noisy images
    tskel = time.time()
    comets_only = skimage.morphology.skeletonize(comets_only)
    final_list.append(comets_only)
    print(f"      Frame {i} -> Total: {time.time()-t0:.2f}s | Skeletonize part: {time.time()-tskel:.2f}s")

# 3. Vector Generation (KDTree) Check
print("[3/5] Calculating vectors via KDTree...")
t_vec = time.time()
vec1 = functionname(final_list[0], final_list[1], 10)
vec2 = functionname(final_list[1], final_list[2], 10)
vec3 = functionname(final_list[2], final_list[3], 10)
vec4 = functionname(final_list[3], final_list[4], 10)
print(f"      Vector calculation took: {time.time()-t_vec:.2f}s")
print(f"      Vector counts -> vec1: {len(vec1)}, vec2: {len(vec2)}, vec3: {len(vec3)}")

# 4. Hashing and Path Tracking Check
print("[4/5] Running path tracking lookups...")
t_track = time.time()
valid_comets = []
vec2_dict = {}
for j in vec2:
    start_key = (j[0], j[1])
    if start_key not in vec2_dict: vec2_dict[start_key] = []
    vec2_dict[start_key].append(j)

vec3_dict = {}
for k in vec3:
    start_key = (k[0], k[1])
    if start_key not in vec3_dict: vec3_dict[start_key] = []
    vec3_dict[start_key].append(k)

for i in vec1:
    x1, y1 = i[2], i[3]
    if (x1, y1) in vec2_dict:
        for j in vec2_dict[(x1, y1)]:
            x3, y3 = j[2], j[3]
            if (x3, y3) in vec3_dict:
                for k in vec3_dict[(x3, y3)]:
                    valid_comets.append((i, j, k))
print(f"      Tracking completed in: {time.time()-t_track:.2f}s")
print(f"      Verified paths found: {len(valid_comets)}")

# 5. Video Generation Check
print("[5/5] Generating output video...")
t_vid = time.time()
if len(valid_comets) > 0:
    frames = paint_sky(valid_comets, 10)
    matrix_movie(frames, "output_file2.mp4", fps=1, scalar=255)
    print(f"      Video saved in: {time.time()-t_vid:.2f}s")
else:
    print("      ⚠️ No verified paths found, skipping video generation.")

print(f"--- DIAGNOSTICS COMPLETE. TOTAL TIME: {time.time()-start_total:.2f}s ---")